# In-Hospital Mortality Prediction in ICU Patients with Acute Kidney Injury (MIMIC-III)

**Author:** Alejandro Vázquez Alonso
**Context:** Master's in AI and Big Data in Health (UAB, 2025-2027)
**Dataset:** MIMIC-III — ICU cohort with Acute Kidney Injury (AKI, KDIGO criteria)
**Target:** IHM (in-hospital mortality), binary — 1 = deceased, 0 = survived

---

### Origin and scope

This pipeline was developed individually as part of a group assignment (4 members) for
Module 2 of the Master's program. The code below is the author's own individual work
product; the group's joint oral presentation and combined submission are separate
deliverables not reproduced here.

### Data access notice

MIMIC-III is a credentialed clinical database (PhysioNet, requires CITI training completion
and a signed Data Use Agreement). **This repository does not include any patient data.**
To reproduce this pipeline, obtain your own authorized access at
[physionet.org/content/mimiciii](https://physionet.org/content/mimiciii/) and place the
resulting CSV at `data/ihm_aki.csv`.

### Notebook structure
1. Imports
2. Data loading and initial inspection
3. Exploratory Data Analysis (EDA)
4. Missing values analysis
5. Preprocessing (missingness indicators, unit correction, physiological range cleaning)
6. Train/test split
7. Imputation and scaling (fit on train only)
8. Class imbalance handling
9. Modeling — Logistic Regression, Random Forest (elbow test + 5-fold CV), XGBoost
10. Model comparison and threshold selection (weighted Youden Index, then F1+AUROC refinement)
11. Final evaluation and overfitting analysis
12. Interpretability


## 1. Imports

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, balanced_accuracy_score,
    confusion_matrix
)

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

print('Librerías importadas correctamente.')

## 2. Data loading and initial inspection

Dataset from MIMIC-III, containing clinical variables for ICU patients with AKI,
summarized as statistics (max, mean, min) over the ICU stay.

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/alejandroava/aki-csv/ihm_aki.csv')
print(f'Dimensiones: {df.shape[0]} pacientes x {df.shape[1]} variables')
print(df.dtypes)

In [ ]:

print('Primeras 10 filas (dataset sin procesar):')
df.head(10)

In [ ]:

df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Target variable distribution

In [ ]:
# --- 3.1 Distribución de la variable objetivo IHM ---
#
# Tasa de mortalidad del 27.4% → desbalanceo moderado (ratio 2.6:1).


ihm_counts = df['IHM'].value_counts()
ihm_pct    = df['IHM'].value_counts(normalize=True) * 100

print('Distribución de IHM:')
print(f'  Supervivientes (0): {ihm_counts[0]} ({ihm_pct[0]:.1f}%)')
print(f'  Fallecidos    (1): {ihm_counts[1]} ({ihm_pct[1]:.1f}%)')
print(f'  Ratio desbalanceo: {ihm_counts[0]/ihm_counts[1]:.2f}:1')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Superviviente (0)', 'Fallecido (1)'], ihm_counts.values,
       color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('Distribución variable objetivo (IHM)')
ax.set_ylabel('N pacientes')
for i, v in enumerate(ihm_counts.values):
    ax.text(i, v + 20, f'{v}\n({ihm_pct.values[i]:.1f}%)',
            ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### 3.2 Correlation of each variable with IHM

In [ ]:
# --- 3.2 Correlación de cada variable con IHM ---
#
# Calculamos la correlación de Pearson antes de entrenar ningún modelo.
# Esto permite verificar después que los modelos priorizan variables
# clínicamente coherentes.
#
# Correlación negativa: cuando la variable sube, la mortalidad baja
# Correlación positiva: cuando la variable sube, la mortalidad sube

numeric_cols = [c for c in df.select_dtypes(include='number').columns
                if c not in ['Unnamed: 0', 'IHM', 'gender_M']]

corr_ihm = df[numeric_cols + ['IHM']].corr()['IHM'].drop('IHM')
corr_ihm = corr_ihm.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['tomato' if v > 0 else 'steelblue' for v in corr_ihm.values]
corr_ihm.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de Pearson con IHM (ordenado por valor absoluto)')
ax.set_xlabel('Correlación')
plt.tight_layout()
plt.show()

print('Top 10 variables más correlacionadas con IHM:')
print(corr_ihm.head(10).round(3))

### 3.3 Multicollinearity among predictors

In [ ]:
# --- 3.4 Matriz de correlación entre variables predictoras ---
#
# Detectamos multicolinealidad: pares de variables casi redundantes.
# Esperamos correlaciones altas entre estadísticos del mismo parámetro
# (wbc_max con wbc_mean, bun_max con bun_mean...).
#
# Esto justifica L2 (no L1) en Logistic Regression:
# L2 estabiliza coeficientes sin eliminar variables.
# L1 eliminaría gcs_min (r=-0.282 con IHM, 5º predictor más fuerte).
# RF y XGBoost manejan la multicolinealidad automáticamente.

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False,
            cmap='coolwarm', center=0, ax=ax, linewidths=0.3)
ax.set_title('Matriz de correlación entre variables predictoras')
plt.tight_layout()
plt.show()

print('Pares con correlación > 0.85 (multicolinealidad severa):')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i,j]
        if abs(r) > 0.85:
            print(f'  {corr_matrix.columns[i]} — '
                  f'{corr_matrix.columns[j]}: r = {r:.3f}')

## 4. Missing values analysis

In [ ]:
missing     = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'N missing': missing, '% missing': missing_pct})
missing_df  = missing_df[missing_df['N missing'] > 0].sort_values(
    '% missing', ascending=False)

print('Variables con valores faltantes:')
print(missing_df)
print(f'\nFilas con al menos un NaN: '
      f'{df.isnull().any(axis=1).sum()} '
      f'({df.isnull().any(axis=1).sum()/len(df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(missing_df.index, missing_df['% missing'],
        color=['tomato' if p > 20 else 'steelblue'
               for p in missing_df['% missing']],
        edgecolor='black')
ax.set_xlabel('% de valores faltantes')
ax.set_title('Missingness por variable')
for i, (idx, row) in enumerate(missing_df.iterrows()):
    ax.text(row['% missing'] + 0.3, i,
            f"{row['% missing']}%", va='center')
plt.tight_layout()
plt.show()

print('\nClasificación del tipo de missingness:')
print('  temp      (34.5%): MNAR — no medida = sin alteración térmica relevante')
print('  bilirubin (18.5%): MNAR — no medida = sin sospecha de daño hepático')
print('  fio2      (11.2%): MAR  — sin soporte O2 o no registrada en MIMIC-III')
print('  pao2      (3.75%): MAR  — gasometría no siempre disponible')

## 5. Preprocessing

Decisions and rationale:
- **No irrelevant columns to drop** — dataset already clean (no numeric index, no redundant
  dummy column: only `gender_F` is present, avoiding the dummy variable trap).
- **Missingness indicators** created for `temp`, `bilirubin`, `fio2` before imputing — since
  missingness itself may be informative (MNAR for `temp`/`bilirubin`, MAR for `fio2`/`pao2`).
- **FiO2 unit correction**: values in 0-1 range are decimal-formatted percentages (×100);
  values in 1-21 are physiologically impossible (minimum room-air FiO2 is 21%) and are
  treated as data-entry errors → converted to NaN.
- **Physiological range cleaning** for heart rate, white blood cell count, and sodium —
  implausible values converted to NaN before imputation.

### 5.1 Column check

In [ ]:
# --- Paso 1: Verificar columnas innecesarias ---
#
# Este dataset ya viene sin índice numérico (Unnamed: 0)
# y sin la columna redundante gender_M.
# Solo tiene gender_F correctamente codificada:
#   gender_F = 1 → mujer | gender_F = 0 → hombre
# No hay dummy variable trap — no es necesario eliminar nada.

df_clean = df.copy()

print(f'Columnas del dataset: {df_clean.shape[1]}')
print('No se requiere eliminar columnas — dataset ya limpio.')

### 5.2 Missingness indicators

In [ ]:
# --- Paso 2: Indicadores de missingness ---
#


for col in ['temp', 'bilirubin', 'fio2']:
    df_clean[f'{col}_missing'] = df_clean[col].isna().astype(int)
    n = df_clean[f'{col}_missing'].sum()
    print(f'{col}_missing: {n} pacientes con dato ausente '
          f'({n/len(df_clean)*100:.1f}%)')

### 5.3 FiO2 unit and range correction

In [ ]:
# --- Paso 3: Corregir errores de escala en FiO2 ---
#
# FiO2 debe estar en % (21-100%). Dos tipos de error:
#
# A) Valores 0-1: formato decimal → multiplicar x100
# B) Valores 1-21: imposibles (mínimo fisiológico = 21% aire ambiente)
#    Son errores de registro sin causalidad biológica → NaN

print('FiO2 antes de corregir:')
print(f'  Valores 0-1 (decimal): '
      f'{((df_clean["fio2"] > 0) & (df_clean["fio2"] < 1)).sum()}')
print(f'  Valores 1-21 (imposibles): '
      f'{((df_clean["fio2"] >= 1) & (df_clean["fio2"] < 21)).sum()}')

df_clean.loc[df_clean['fio2'] < 1, 'fio2'] = \
    df_clean.loc[df_clean['fio2'] < 1, 'fio2'] * 100

df_clean.loc[
    (df_clean['fio2'] > 1) & (df_clean['fio2'] < 21), 'fio2'] = np.nan

print('\nFiO2 tras corrección:')
print(f'  min={df_clean["fio2"].min():.1f}% | '
      f'max={df_clean["fio2"].max():.1f}% | '
      f'mediana={df_clean["fio2"].median():.1f}%')

### 5.4 Physiological range cleaning (HR, WBC, Sodium)

In [ ]:


criterios = {
    'HR':    (['hr_max', 'hr_mean', 'hr_min'],    30,  220),
    'WBC':   (['wbc_max', 'wbc_mean', 'wbc_min'],  1,  100),
    'Sodio': (['sod_max', 'sod_mean', 'sod_min'], 110, 170)
}

for nombre, (cols, lo, hi) in criterios.items():
    print(f'\n{nombre} [{lo}, {hi}]:')
    for col in cols:
        n = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
        df_clean.loc[
            (df_clean[col] < lo) | (df_clean[col] > hi), col] = np.nan
        print(f'  {col}: {n} valores imposibles → NaN')

### 5.5 Columns to impute (final names)

In [ ]:
# --- Paso 5: Definir columnas para imputar ---
# Se definen aquí con los nombres FINALES (después del renombrado).
# La imputación real se ejecutará después del split.

cols_to_impute = [
    'temp', 'bilirubin', 'fio2',
    'pao2_max', 'pao2_mean', 'pao2_min',
    'wbc_max', 'wbc_mean', 'wbc_min',
    'hr_max', 'hr_mean', 'hr_min',
    'sod_max', 'sod_mean', 'sod_min'
]

print('Columnas definidas para imputación:')
print(cols_to_impute)

In [ ]:
df_clean = df_clean.rename(columns={
    'max pao2':  'pao2_max',
    'mean pao2': 'pao2_mean',
    'min pao2':  'pao2_min'
})
print('Columnas renombradas correctamente.')

## 6. Train/test split

Stratified 80/20 split, performed **before** imputation and scaling to avoid data leakage.

In [ ]:
X = df_clean.drop(columns=['IHM'])
y = df_clean['IHM']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'Train: {X_train.shape[0]} pacientes '
      f'({y_train.mean()*100:.1f}% mortalidad)')
print(f'Test:  {X_test.shape[0]} pacientes '
      f'({y_test.mean()*100:.1f}% mortalidad)')
print(f'\nstratify=y → misma proporción en train y test ✓')

## 7. Imputation and scaling

Median imputation and StandardScaler, both **fit only on train** and applied to test —
avoids leakage from test statistics into the transformation.

In [ ]:
# Imputación + Escalado después del split
# fit SOLO sobre train — evita data leakage

# X_train ya es DataFrame con nombres de columnas correctos
# No necesitamos convertir nada

# Verificar NaN antes
print(f'NaN en X_train antes de imputar: {X_train.isnull().sum().sum()}')
print(f'NaN en X_test  antes de imputar: {X_test.isnull().sum().sum()}')

# Copias para no modificar los originales
X_train_imp = X_train.copy()
X_test_imp  = X_test.copy()

# Imputar por mediana — fit SOLO sobre train
imputer = SimpleImputer(strategy='median')
X_train_imp[cols_to_impute] = imputer.fit_transform(
    X_train_imp[cols_to_impute])
X_test_imp[cols_to_impute]  = imputer.transform(
    X_test_imp[cols_to_impute])

# Verificar NaN después
print(f'\nNaN en X_train después de imputar: {X_train_imp.isnull().sum().sum()}')
print(f'NaN en X_test  después de imputar: {X_test_imp.isnull().sum().sum()}')

# Escalar — fit SOLO sobre train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# Verificación final
nan_train_final = pd.DataFrame(X_train_scaled).isnull().sum().sum()
nan_test_final  = pd.DataFrame(X_test_scaled).isnull().sum().sum()
print(f'\nNaN en X_train_scaled: {nan_train_final}')
print(f'NaN en X_test_scaled:  {nan_test_final}')

if nan_train_final == 0 and nan_test_final == 0:
    print('\nImputación y escalado correctos — listo para modelar ✓')
else:
    print('\nATENCIÓN: todavía quedan NaN')

print(f'\nMedianas aprendidas del train:')
for col, med in zip(cols_to_impute, imputer.statistics_):
    print(f'  {col}: {med:.2f}')

## 8. Class imbalance handling

Mortality rate 27.4% (ratio ~2.64:1). `class_weight='balanced'` used for Logistic Regression
and Random Forest rather than SMOTE, to avoid generating synthetic clinically-implausible
patient records in a moderate-size dataset.

In [ ]:
# Verificamos los pesos que se aplicarán
n_total = len(y_train)
n_neg   = (y_train == 0).sum()
n_pos   = (y_train == 1).sum()

peso_neg = n_total / (2 * n_neg)
peso_pos = n_total / (2 * n_pos)

print('Distribución de clases en train:')
print(f'  Supervivientes (0): {n_neg} → peso = {peso_neg:.2f}')
print(f'  Fallecidos    (1): {n_pos} → peso = {peso_pos:.2f}')
print(f'\nEl error en un fallecido pesa {peso_pos/peso_neg:.1f}x '
      f'más que en un superviviente durante el entrenamiento')

## 9. Modeling — Logistic Regression (baseline)

**Why L2 and not L1?** The correlation matrix (section 3.3) shows several pairs of
near-redundant variables (e.g. `wbc_max`/`wbc_mean`, `bun_max`/`bun_mean`). L1 would tend to
zero out correlated predictors — including `gcs_min`, the 5th strongest predictor of IHM by
correlation. L2 stabilizes coefficients without eliminating clinically relevant variables.

In [ ]:
lr = LogisticRegression(
    penalty='l2',
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    random_state=SEED
)
lr.fit(X_train_scaled, y_train)

y_prob_lr       = lr.predict_proba(X_test_scaled)[:, 1]
y_prob_lr_train = lr.predict_proba(X_train_scaled)[:, 1]

auroc_lr       = roc_auc_score(y_test,  y_prob_lr)
auroc_lr_train = roc_auc_score(y_train, y_prob_lr_train)

print(f'Logistic Regression (L2)')
print(f'  AUROC Train: {auroc_lr_train:.4f}')
print(f'  AUROC Test:  {auroc_lr:.4f}')
print(f'  Diferencia:  {auroc_lr_train - auroc_lr:+.4f}')

## 10. Modeling — Random Forest

### 10.1 Elbow test: AUROC vs n_estimators (50 to 600, step 50)

In [ ]:
# --- Test del codo: curva AUROC vs n_estimators ---
#
# Curva completa de 50 en 50 árboles.
# El punto donde la curva test se aplana = número óptimo.

resultados_codo = []
for n in range(50, 650, 50):
    rf_tmp = RandomForestClassifier(
        n_estimators=n, max_depth=10,
        min_samples_leaf=5, class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )
    rf_tmp.fit(X_train_scaled, y_train)
    auroc_tr = roc_auc_score(
        y_train, rf_tmp.predict_proba(X_train_scaled)[:,1])
    auroc_te = roc_auc_score(
        y_test,  rf_tmp.predict_proba(X_test_scaled)[:,1])
    resultados_codo.append((n, auroc_tr, auroc_te))
    print(f'  n={n:3d}: train={auroc_tr:.4f} | test={auroc_te:.4f}')

df_codo = pd.DataFrame(
    resultados_codo, columns=['n', 'auroc_train', 'auroc_test'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_codo['n'], df_codo['auroc_train'],
        label='AUROC Train', color='steelblue', lw=2, marker='o', ms=4)
ax.plot(df_codo['n'], df_codo['auroc_test'],
        label='AUROC Test', color='tomato', lw=2, marker='o', ms=4)
ax.set_xlabel('Número de árboles')
ax.set_ylabel('AUROC')
ax.set_title('Test del codo — Random Forest: AUROC vs n_estimators')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mejor_n_codo = int(df_codo.loc[df_codo['auroc_test'].idxmax(), 'n'])
print(f'\nMejor n_estimators según test del codo: {mejor_n_codo}')

### 10.2 5-fold cross-validation to confirm n_estimators

In [ ]:
# --- CV 5-Fold para confirmar la elección ---

print('CV 5-Fold — comparativa de configuraciones:')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}

for n in [100, 300, 500]:
    rf_tmp = RandomForestClassifier(
        n_estimators=n, max_depth=10,
        min_samples_leaf=5, class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )
    scores = cross_val_score(
        rf_tmp, X_train_scaled, y_train,
        cv=cv, scoring='balanced_accuracy'
    )
    cv_results[n] = (scores.mean(), scores.std())
    print(f'  n={n:3d}: BA={scores.mean():.4f} ± {scores.std():.4f}')

mejor_n = max(cv_results, key=lambda k: cv_results[k][0])
print(f'\nConfiguración seleccionada: n_estimators = {mejor_n}')

### 10.3 Final Random Forest model

In [ ]:
# Entrenamiento final del Random Forest
rf = RandomForestClassifier(
    n_estimators=mejor_n,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

y_prob_rf       = rf.predict_proba(X_test_scaled)[:, 1]
y_prob_rf_train = rf.predict_proba(X_train_scaled)[:, 1]

auroc_rf       = roc_auc_score(y_test,  y_prob_rf)
auroc_rf_train = roc_auc_score(y_train, y_prob_rf_train)

print(f'Random Forest ({mejor_n} árboles)')
print(f'  AUROC Train: {auroc_rf_train:.4f}')
print(f'  AUROC Test:  {auroc_rf:.4f}')
print(f'  Diferencia:  {auroc_rf_train - auroc_rf:+.4f}')

## 11. Modeling — XGBoost

**Why XGBoost over SVM or neural networks?** SVM scales poorly (O(n²)) with ~3,550 patients
and offers low clinical interpretability. Neural networks are prone to overfitting on a
dataset of this size without a clear accuracy gain over tree ensembles, which additionally
provide native, interpretable feature importance.

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=SEED
)
xgb.fit(X_train_scaled, y_train)

y_prob_xgb       = xgb.predict_proba(X_test_scaled)[:, 1]
y_prob_xgb_train = xgb.predict_proba(X_train_scaled)[:, 1]

auroc_xgb       = roc_auc_score(y_test,  y_prob_xgb)
auroc_xgb_train = roc_auc_score(y_train, y_prob_xgb_train)

print(f'XGBoost')
print(f'  AUROC Train: {auroc_xgb_train:.4f}')
print(f'  AUROC Test:  {auroc_xgb:.4f}')
print(f'  Diferencia:  {auroc_xgb_train - auroc_xgb:+.4f}')

## 12. Model comparison and threshold selection

### 12.1 First pass — Weighted Youden Index

The 0.5 threshold has no clinical justification for this problem. False negatives
(predicting survival for a patient who dies) carry a much higher clinical cost than false
positives (unnecessary intensive monitoring) — the same asymmetry addressed in a prior
assignment (fetal wellbeing classification, CTG data), where sensitivity was weighted 3×
over specificity in the Youden Index.

In [ ]:
# Seleccionar el modelo con mejor AUROC
aurocs = {
    'LR':  (auroc_lr,  y_prob_lr),
    'RF':  (auroc_rf,  y_prob_rf),
    'XGB': (auroc_xgb, y_prob_xgb)
}
mejor_modelo = max(aurocs, key=lambda k: aurocs[k][0])
y_prob_final = aurocs[mejor_modelo][1]

print(f'Modelo seleccionado: {mejor_modelo} '
      f'(AUROC = {aurocs[mejor_modelo][0]:.4f})')

# Calcular Youden estándar y ponderado sobre la curva ROC
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob_final)

# Youden estándar: J = Sensibilidad + Especificidad - 1
J_standard = tpr - fpr
idx_std     = np.argmax(J_standard)
tau_std     = thresholds_roc[idx_std]

# Youden ponderado: sensibilidad vale 3x (Actividad 6 — CTG)
J_ponderado = 3 * tpr - fpr
idx_pond    = np.argmax(J_ponderado)
tau_pond    = thresholds_roc[idx_pond]

print(f'\nYouden estándar:   tau={tau_std:.4f} | '
      f'Sens={tpr[idx_std]*100:.1f}% | '
      f'Espec={(1-fpr[idx_std])*100:.1f}%')
print(f'Youden ponderado:  tau={tau_pond:.4f} | '
      f'Sens={tpr[idx_pond]*100:.1f}% | '
      f'Espec={(1-fpr[idx_pond])*100:.1f}%')

# Visualización sobre la curva ROC
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='tomato', lw=2,
        label=f'{mejor_modelo} (AUC={aurocs[mejor_modelo][0]:.3f})')
ax.scatter(fpr[idx_std], tpr[idx_std], color='steelblue', s=120, zorder=5,
           label=f'Youden estándar   (tau={tau_std:.2f}, '
                 f'Sens={tpr[idx_std]*100:.0f}%)')
ax.scatter(fpr[idx_pond], tpr[idx_pond], color='orange', s=120, zorder=5,
           label=f'Youden ponderado  (tau={tau_pond:.2f}, '
                 f'Sens={tpr[idx_pond]*100:.0f}%)')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
ax.set_ylabel('Sensibilidad')
ax.set_title(f'Curva ROC con puntos óptimos de Youden — {mejor_modelo}')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Tabla de métricas para distintos umbrales
tau_std_r  = round(tau_std,  2)
tau_pond_r = round(tau_pond, 2)

umbrales = sorted(set(
    [0.10, 0.20, 0.30, tau_pond_r, tau_std_r, 0.50, 0.60, 0.70]
))
filas_tau = []

for tau in umbrales:
    y_pred_tau = (y_prob_final >= tau).astype(int)
    rep = classification_report(
        y_test, y_pred_tau, output_dict=True, zero_division=0)
    fn  = int((y_test==1).sum() - y_pred_tau[y_test==1].sum())
    fp  = int(y_pred_tau[y_test==0].sum())
    acc = (y_pred_tau == y_test).mean()
    marcador = '← YOUDEN PONDERADO' if tau == tau_pond_r else \
               '← YOUDEN ESTANDAR'  if tau == tau_std_r  else ''
    filas_tau.append({
        'tau':           tau,
        'Accuracy':      round(acc, 3),
        'F1':            round(f1_score(
            y_test, y_pred_tau, zero_division=0), 3),
        'Sensibilidad':  round(rep['1']['recall'], 3),
        'Especificidad': round(rep['0']['recall'], 3),
        'FN':            fn,
        'FP':            fp,
        'Nota':          marcador
    })

df_tau = pd.DataFrame(filas_tau)
print(f'Métricas por umbral — modelo {mejor_modelo}:')
print(df_tau.to_string(index=False))

In [ ]:
TAU = round(tau_pond, 2)
y_pred_final = (y_prob_final >= TAU).astype(int)

print(f'=== EVALUACIÓN FINAL — {mejor_modelo} (tau = {TAU}) ===')
print(f'AUROC:             {roc_auc_score(y_test, y_prob_final):.4f}')
print(f'F1-score:          {f1_score(y_test, y_pred_final):.4f}')
print(f'Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_final):.4f}')
print(f'\nClassification Report:')
print(classification_report(
    y_test, y_pred_final,
    target_names=['Superviviente', 'Fallecido']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=['Superviviente', 'Fallecido'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Matriz de confusión — {mejor_modelo} (tau={TAU})')
plt.tight_layout()
plt.show()

print(f'\n>>> UMBRAL DECLARADO PARA DATOS RESERVADOS: tau = {TAU} <<<')

### 12.2 Refinement — combined AUROC + F1 score across all thresholds

The weighted-Youden threshold above prioritized sensitivity aggressively (94% sensitivity,
64% specificity), at the cost of a high false-positive rate. Since the assignment evaluates
models on both AUROC and F1, a second pass searches, for each model, the threshold that
maximizes F1, then ranks models by the mean of AUROC and optimal F1 — balancing
discrimination and threshold-dependent performance rather than optimizing for sensitivity
alone.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_curve

umbrales = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40,
            0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 0.90]

for nombre_m, y_prob_m, auroc_m in [
    ('LOGISTIC REGRESSION L2', y_prob_lr,  auroc_lr),
    ('RANDOM FOREST',          y_prob_rf,  auroc_rf),
    ('XGBOOST',                y_prob_xgb, auroc_xgb)
]:
    # Calcular tau óptimo para este modelo
    prec_c, rec_c, thr_c = precision_recall_curve(y_test, y_prob_m)
    f1s_c   = 2*(prec_c*rec_c)/(prec_c+rec_c+1e-8)
    tau_opt = round(thr_c[np.argmax(f1s_c)], 2)

    print(f'\n{"="*80}')
    print(f'  {nombre_m}  —  AUROC: {auroc_m:.4f}  |  Tau óptimo: {tau_opt}')
    print(f'{"="*80}')
    print(f'{"tau":>5} | {"Accuracy":>9} | {"F1":>7} | '
          f'{"Sensib.":>8} | {"Especif.":>9} | {"FN":>4} | {"FP":>4}')
    print('-'*65)

    for tau in umbrales:
        y_pred_tau = (y_prob_m >= tau).astype(int)
        rep  = classification_report(
            y_test, y_pred_tau,
            output_dict=True, zero_division=0)
        acc  = accuracy_score(y_test, y_pred_tau)
        f1   = f1_score(y_test, y_pred_tau, zero_division=0)
        sens = rep['1']['recall']
        espe = rep['0']['recall']
        fn   = int((y_test==1).sum() - y_pred_tau[y_test==1].sum())
        fp   = int(y_pred_tau[y_test==0].sum())
        marca = ' <- OPTIMO' if tau == tau_opt else ''

        print(f'{tau:>5.2f} | {acc*100:>8.1f}% | {f1:>7.4f} | '
              f'{sens*100:>7.1f}% | {espe*100:>8.1f}% | '
              f'{fn:>4} | {fp:>4}{marca}')

print('\nAUROC es independiente del umbral — mismo valor en todas las filas.')

In [ ]:
# Selección del mejor modelo y umbral según combinación AUROC + F1
# El enunciado evalúa ambas métricas — buscamos el óptimo conjunto

from sklearn.metrics import precision_recall_curve, accuracy_score

print('=== SELECCIÓN DEL MEJOR MODELO Y UMBRAL ===')
print('Criterio: maximizar F1 (el AUROC no depende del umbral)\n')

resultados = []

for nombre_m, y_prob_m, auroc_m in [
    ('Logistic Regression L2', y_prob_lr,  auroc_lr),
    ('Random Forest',          y_prob_rf,  auroc_rf),
    ('XGBoost',                y_prob_xgb, auroc_xgb)
]:
    # Tau que maximiza F1
    prec_c, rec_c, thr_c = precision_recall_curve(y_test, y_prob_m)
    f1s_c   = 2*(prec_c*rec_c)/(prec_c+rec_c+1e-8)
    idx_opt = np.argmax(f1s_c)
    tau_opt = thr_c[idx_opt]

    y_pred_opt = (y_prob_m >= tau_opt).astype(int)
    rep  = classification_report(
        y_test, y_pred_opt, output_dict=True, zero_division=0)
    f1_opt   = f1_score(y_test, y_pred_opt, zero_division=0)
    acc_opt  = accuracy_score(y_test, y_pred_opt)
    sens_opt = rep['1']['recall']
    espe_opt = rep['0']['recall']
    fn_opt   = int((y_test==1).sum() - y_pred_opt[y_test==1].sum())

    # Score combinado: media ponderada AUROC + F1
    # Mismo peso a ambas porque el enunciado evalúa las dos por igual
    score_combinado = (auroc_m + f1_opt) / 2

    resultados.append({
        'Modelo':           nombre_m,
        'AUROC':            round(auroc_m, 4),
        'F1_optimo':        round(f1_opt, 4),
        'Score_combinado':  round(score_combinado, 4),
        'tau_optimo':       round(tau_opt, 2),
        'Accuracy':         round(acc_opt, 4),
        'Sensibilidad':     round(sens_opt, 4),
        'Especificidad':    round(espe_opt, 4),
        'FN':               fn_opt,
        'y_prob':           y_prob_m
    })

# Tabla comparativa
print(f'{"Modelo":<24} {"AUROC":>7} {"F1 opt":>8} {"Score":>7} '
      f'{"tau":>6} {"Sensib.":>8} {"Especif.":>9} {"FN":>4}')
print('-'*80)

for r in sorted(resultados, key=lambda x: x['Score_combinado'], reverse=True):
    print(f'{r["Modelo"]:<24} {r["AUROC"]:>7.4f} {r["F1_optimo"]:>8.4f} '
          f'{r["Score_combinado"]:>7.4f} {r["tau_optimo"]:>6.2f} '
          f'{r["Sensibilidad"]*100:>7.1f}% {r["Especificidad"]*100:>8.1f}% '
          f'{r["FN"]:>4}')

# Seleccionar ganador
ganador = max(resultados, key=lambda x: x['Score_combinado'])

print(f'\n{"="*60}')
print(f'MODELO GANADOR:  {ganador["Modelo"]}')
print(f'  AUROC:         {ganador["AUROC"]}')
print(f'  F1 óptimo:     {ganador["F1_optimo"]}')
print(f'  Score (media): {ganador["Score_combinado"]}')
print(f'  Tau declarado: {ganador["tau_optimo"]}')
print(f'{"="*60}')

# Guardar para la evaluación final
mejor_modelo  = ganador['Modelo']
y_prob_final  = ganador['y_prob']
TAU_FINAL     = ganador['tau_optimo']

TAU_FINAL = round(float(ganador['tau_optimo']), 2)
print(f'\n>>> UMBRAL DECLARADO PARA DATOS RESERVADOS: tau = {TAU_FINAL} <<<')

## 13. Overfitting analysis

Logistic Regression shows minimal train-test gap (well regularized). Random Forest and
XGBoost both show a larger apparent gap, consistent with tree ensembles fitting training
data closely; the 5-fold CV balanced accuracy for Random Forest confirms the test-set
performance is not an artifact of the split.

In [ ]:
print('=== ANÁLISIS DE OVERFITTING ===\n')
print(f'{"Modelo":<28} {"AUROC Train":>12} {"AUROC Test":>11} '
      f'{"Diferencia":>11}  Diagnóstico')
print('-'*80)

for nombre, auroc_tr, auroc_te in [
    ('Logistic Regression L2', auroc_lr_train,  auroc_lr),
    ('Random Forest',          auroc_rf_train,  auroc_rf),
    ('XGBoost',                auroc_xgb_train, auroc_xgb)
]:
    diff = auroc_tr - auroc_te
    if diff > 0.05:
        diagnostico = 'Aparente — verificar con CV'
    elif diff > 0.02:
        diagnostico = 'Leve — aceptable'
    else:
        diagnostico = 'Sin overfitting significativo'
    print(f'{nombre:<28} {auroc_tr:>12.4f} {auroc_te:>11.4f} '
          f'{diff:>+11.4f}  {diagnostico}')

print(f'\nCV 5-Fold BA Random Forest: {cv_results[mejor_n][0]:.4f} '
      f'± {cv_results[mejor_n][1]:.4f}')
print('BA_cv ≈ BA_test → generalización correcta confirmada')

## 14. Interpretability

Feature importance comparison between Random Forest and XGBoost.

In [ ]:
# Importancia de variables — RF vs XGBoost
feature_names = X.columns.tolist()

imp_rf  = pd.Series(
    rf.feature_importances_, index=feature_names
).sort_values(ascending=False).head(15)
imp_xgb = pd.Series(
    xgb.feature_importances_, index=feature_names
).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

imp_rf.sort_values().plot(
    kind='barh', ax=axes[0], color='seagreen', edgecolor='black')
axes[0].set_title('Top 15 variables — Random Forest')
axes[0].set_xlabel('Importancia (mean decrease impurity)')

imp_xgb.sort_values().plot(
    kind='barh', ax=axes[1], color='tomato', edgecolor='black')
axes[1].set_title('Top 15 variables — XGBoost')
axes[1].set_xlabel('Importancia (gain)')

plt.suptitle('Comparativa importancia de variables — RF vs XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
# Verificación de coherencia clínica y feature engineering
top10_rf  = set(imp_rf.head(10).index)
top10_xgb = set(imp_xgb.head(10).index)
coincidencias = top10_rf.intersection(top10_xgb)

print('Variables en el Top 10 de AMBOS modelos:')
for v in sorted(coincidencias):
    print(f'  {v}')

print('\nInterpretación clínica de los principales predictores:')
print('  GCS (max/mean/min): encefalopatía urémica — nivel de consciencia')
print('  BUN (max/mean):     acumulación de toxinas nitrogenadas en AKI')
print('  bic_min:            acidosis metabólica — riñón sin regenerar bicarbonato')
print('  bp_min:             hipotensión / shock — inestabilidad hemodinámica')
print('  age:                menor reserva funcional renal en edad avanzada')

print('\nVerificación feature engineering:')
for var in ['gcs_range', 'bp_range', 'pao2_fio2_ratio']:
    en_rf  = var in imp_rf.index
    en_xgb = var in imp_xgb.index
    print(f'  {var}: RF={en_rf} | XGB={en_xgb}')

## 15. AI use declaration

This pipeline was developed with the assistance of generative AI (Claude, Anthropic) for
code optimization, conceptual explanation of methodological decisions, and validation of
clinical interpretation against established criteria (KDIGO for AKI). All final
methodological decisions and clinical interpretation were reviewed and validated by the
author, drawing on 9+ years of clinical experience as a physiotherapist.